## M505 Intro to AI and Machine Learning

### Name: Manan Girishbhai Chauhan (GH1047553)

### Problem Statement:-
- As the newly appointed data scientist at e-commerce company the business problem is low conversion rates only 15% of website result in purchases and leading to lost the revenue. this is important because improving prediction of intention can be enable personalized marketing, advertisements and better user experience sales can be increasing by 20-30 %.
- This is a binary classification ML task to predict revenue based on features and solving this benefits the company by optimizing resources and boosting profits.

### Data Source:-
https://www.kaggle.com/datasets/imakash3011/online-shoppers-purchasing-intention-dataset?resource=download

### Feature Explanation:-
- Administrative: Number of admin pages
- Administrative_Duration: Total time spent on admin pages
- Informational: Number of  informational pages.
- Informational_Duration: Total time spent on informational pages.
- ProductRelated : Product related pages numbers
- ProductRelated_Duration: Total time spent on product related pages longer times consideration.
- BounceRates: Percentage of visitor who leave after viewing single page
- ExitRates : Percentage pf pageview where the page was last in the session.
- PageValues: Average economic value of page visited before purchase
- Specialday: Numeric indicatrtor of special days
- Feature: Description or possible value
- Month: Month of session
- Operating systems: Operating system of user
- Region : Geographic region of the vistiors.
- TrafficType: How the visitor arrived
- VisitorType: Type of visitor returnig,newm other.
- Weekend: True if session on weekend
- Revenue: Target feature binary label true or false.


### Step 1 : Importing models and libraries
- First we will import libraries, models and accuracy and model evaluation libraries

In [1]:
import os
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import average_precision_score

### Step 2 : loading dataset
- in this step we will load the dataset.


In [ ]:
# for reproducibility
np.random.seed(42)
# Load data
dataset = "online_shoppers_intention.csv"
assert os.path.exists(dataset),f"CSV not found at {dataset}"
df = pd.read_csv(dataset)


### Step 3 : Overview of dataset and ensuring data set is clean 
- In this step we will check dataset rows and columns and its datatypes of each column or feature also we will check if data set is clean or not.
- Here we have clean data set and we are also checking if any duplicate values are there or not.

In [ ]:
# Basic info of dataset
print("Shape:-->", df.shape)
print(df.dtypes)
print("Dropped identifier: None")
df = df.drop(columns=["Unnamed: 0"] if "Unnamed: 0" in df.columns else [], axis=1)
print("Dropped duplicates:-->", df.duplicated().sum())
print("Missing values per column:-->\n", df.isnull().sum())


### Step 4: Checking target disribution
- In this step we have to predict the revenue for that we are checking distribution of the revenue column in data set by using value counts function in three categories like class,count,and ratio.

In [ ]:
df['Revenue'] = df['Revenue'].astype(int)
class_balance = pd.DataFrame({
    'class': [0, 1],
    'count': [len(df[df['Revenue'] == 0]), len(df[df['Revenue'] == 1])],
    'ratio': [len(df[df['Revenue'] == 0])/len(df), len(df[df['Revenue'] == 1])/len(df)]
})
display(class_balance)

### Step 5: Boxplot and correlation
- In this step we are setting the size and parameters of visual charts
- We are using boxplot to see a correlation betweem pagevalues and purchase.


In [ ]:
plt.rcParams['figure.figsize'] = (7,5)
plt.rcParams['axes.grid'] = True

In [ ]:
sns.boxplot(x='Revenue', y='PageValues', data=df)
plt.title('Box plot PageValues vs Purchase Intention')
plt.show()

In [ ]:
num_cols = ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
            'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
plt.figure(figsize=(12, 10))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.title('Correlation Heatmap')
plt.show()


### Step 6: Data perprocessing and feature engineering
- In step 6 we are droping the revenue to train model also in other words we can we are dropping target feature to train model perfectly
- In this dataset containes numerical and categorical features hence we are converting categorical features into numerical.
- we are preprocessing dataset using onehotencoder and scaler

In [ ]:
X = df.drop(columns=['Revenue'])
y = df['Revenue']


num_cols = ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
            'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
cat_cols = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline(steps=[
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)
    ])


X_pre = preprocessor.fit_transform(X)
print("Preprocessed shape:", X_pre.shape)

### Step 7 : Model Trainings and validation of model
- In this step we are training the model and we are spliting dataset into training,validation and testing.
- We have used python dictionary to train multiple model at a time.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

models = {
    'LogisticRegression': Pipeline([('pre', preprocessor), ('clf', LogisticRegression(class_weight='balanced', random_state=42))]),
    'DecisionTree': Pipeline([('pre', preprocessor), ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=42))]),
    'RandomForest': Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))]),
    'GradientBoosting': Pipeline([('pre', preprocessor), ('clf', GradientBoostingClassifier(random_state=42))])
}


results = {}
for name, model in models.items():
    model.fit(X_train_sub, y_train_sub)
    y_val_prob = model.predict_proba(X_val)[:, 1]
    results[name] = average_precision_score(y_val, y_val_prob)
print("PR AUC Scores:", results)

best_model = models['RandomForest']


### Step 8 : Hyperparameter tuning  and using GridsearchCV

In [ ]:
param_grid = {'clf__n_estimators': [50, 100, 200], 'clf__max_depth': [None, 10, 20]}
grid_search = GridSearchCV(best_model, param_grid, cv=3, scoring='average_precision')
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print("Best Params:", grid_search.best_params_)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, classification_report

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

print("Test ROC AUC:", roc_auc_score(y_test, y_prob))
print("Test PR AUC:", average_precision_score(y_test, y_prob))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# PR Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
plt.plot(recall, precision)
plt.title('Precision-Recall Curve')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.show()

In [ ]:
importances = pd.Series(best_model.named_steps['clf'].feature_importances_,
                        index=best_model.named_steps['pre'].get_feature_names_out()).sort_values(ascending=False)
sns.barplot(x=importances.values[:12], y=importances.index[:12])
plt.title('Top Feature Importance')
plt.show()

### Final Discussion:-
- Strenghts: This pipline is comprehensive and it compares multiple models and checking RF,PR, AUC. it acieves approximately 90% ROC AUC and high recall. Some identifying key features like pagevalue.

- Limitation: Imbalance data may inflate metrics and no oversampling tested.

- Implications: It can prioritize high intent shoppers and it can also usefull to boost the revenue or profit.

- Recommendation: It can be integrated with website for real time ads and collect more data.

- Informative features- Pagevalues and exitrates means transaction proxy and engagement on page can explainable via importance´.